In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags

df = load_all_snapshots(seasons=[2024])
f = add_discipline_flags(df)
f = f[f["pitch_type"].notna()]

# Per (batter, pitch_type) splits
sw = f[f["is_swing"]]
oz = f[~f["in_zone"]]

by_pitch = pd.DataFrame({
    "pitches": f.groupby(["batter", "pitch_type"]).size(),
    "swings": sw.groupby(["batter", "pitch_type"]).size(),
    "whiffs": sw.groupby(["batter", "pitch_type"])["is_whiff"].sum(),
    "oz_pitches": oz.groupby(["batter", "pitch_type"]).size(),
    "oz_swings": oz.groupby(["batter", "pitch_type"])["is_swing"].sum(),
}).fillna(0)

by_pitch["whiff_pct"] = by_pitch["whiffs"] / by_pitch["swings"]
by_pitch["chase_pct"] = by_pitch["oz_swings"] / by_pitch["oz_pitches"]

# League reference per pitch type
league = pd.DataFrame({
    "lg_whiff": sw.groupby("pitch_type")["is_whiff"].mean(),
    "lg_chase": oz.groupby("pitch_type")["is_swing"].mean(),
})
print(league.round(3).to_string())

            lg_whiff  lg_chase
pitch_type                    
CH             0.293     0.337
CS             0.125     0.308
CU             0.296     0.290
EP             0.020     0.285
FA             0.098     0.216
FC             0.213     0.272
FF             0.189     0.239
FO             0.272     0.427
FS             0.327     0.359
KC             0.329     0.322
KN             0.274     0.250
PO               NaN     0.000
SC             0.232     0.226
SI             0.117     0.246
SL             0.323     0.321
ST             0.297     0.308
SV             0.271     0.292
UN               NaN     0.000


In [2]:
# Vertical thirds of the batter's own zone, plus above and below.
pz = pd.to_numeric(f["plate_z"], errors="coerce")
top = pd.to_numeric(f["sz_top"], errors="coerce")
bot = pd.to_numeric(f["sz_bot"], errors="coerce")
f["z_rel"] = (pz - bot) / (top - bot)

f["zone_band"] = pd.cut(f["z_rel"], bins=[-np.inf, 0, 0.33, 0.67, 1.0, np.inf],
                        labels=["below", "low", "middle", "high", "above"])

swz = f[f["is_swing"]]
by_zone = pd.DataFrame({
    "swings": swz.groupby(["batter", "zone_band"], observed=True).size(),
    "whiffs": swz.groupby(["batter", "zone_band"], observed=True)["is_whiff"].sum(),
})
by_zone["whiff_pct"] = by_zone["whiffs"] / by_zone["swings"]

lg_zone = swz.groupby("zone_band", observed=True)["is_whiff"].mean()
print(lg_zone.round(3).to_string())

zone_band
below     0.532
low       0.198
middle    0.123
high      0.181
above     0.352


In [3]:
from src.data.player_ids import load_player_ids, display_name

MIN_SWINGS_PER_PITCH = 50      # Day 16 territory: below this, whiff% is noise
MIN_SWINGS_PER_ZONE = 40
NOTABLE_SD = 1.0               # how far from league before we say anything

def batter_report(batter_id, names=None):
    lines = []
    name = names.get(batter_id, str(batter_id)) if names is not None else str(batter_id)
    lines.append(f"# {name}")

    # --- pitch type vulnerabilities
    try:
        bp = by_pitch.loc[batter_id]
    except KeyError:
        return "\n".join(lines + ["", "INSUFFICIENT SAMPLE — no pitch data"])

    bp = bp[bp["swings"] >= MIN_SWINGS_PER_PITCH].join(league)
    if len(bp) == 0:
        lines.append("\n## Pitch types\nINSUFFICIENT SAMPLE")
    else:
        bp = bp.copy()
        bp["whiff_vs_lg"] = bp["whiff_pct"] - bp["lg_whiff"]
        lines.append("\n## Pitch types")
        lines.append(f"({len(bp)} pitch types with {MIN_SWINGS_PER_PITCH}+ swings)")
        for pt, r in bp.sort_values("whiff_vs_lg", ascending=False).iterrows():
            marker = ""
            if r["whiff_vs_lg"] > 0.05:
                marker = "  <-- vulnerable"
            elif r["whiff_vs_lg"] < -0.05:
                marker = "  <-- handles well"
            lines.append(
                f"  {pt}: whiff {r['whiff_pct']:.1%} vs league {r['lg_whiff']:.1%} "
                f"({r['whiff_vs_lg']:+.1%}), {int(r['swings'])} swings{marker}")

    # --- zone bands
    try:
        bz = by_zone.loc[batter_id]
        bz = bz[bz["swings"] >= MIN_SWINGS_PER_ZONE]
    except KeyError:
        bz = pd.DataFrame()

    lines.append("\n## Location")
    if len(bz) == 0:
        lines.append("INSUFFICIENT SAMPLE")
    else:
        for band, r in bz.iterrows():
            lg = lg_zone.get(band, np.nan)
            lines.append(
                f"  {band}: whiff {r['whiff_pct']:.1%} vs league {lg:.1%} "
                f"({r['whiff_pct'] - lg:+.1%}), {int(r['swings'])} swings")

    return "\n".join(lines)

ids = load_player_ids([592450, 665742, 660271])   # Judge, Soto, Ohtani
names = display_name(ids)

print(batter_report(592450, names))

# Judge, Aaron

## Pitch types
(7 pitch types with 50+ swings)
  CH: whiff 45.8% vs league 29.3% (+16.5%), 118 swings  <-- vulnerable
  CU: whiff 44.3% vs league 29.6% (+14.7%), 61 swings  <-- vulnerable
  SL: whiff 42.2% vs league 32.3% (+9.9%), 180 swings  <-- vulnerable
  FC: whiff 29.2% vs league 21.3% (+7.9%), 106 swings  <-- vulnerable
  ST: whiff 31.4% vs league 29.7% (+1.7%), 121 swings
  FF: whiff 20.1% vs league 18.9% (+1.1%), 369 swings
  SI: whiff 12.6% vs league 11.7% (+0.9%), 199 swings

## Location
  below: whiff 73.2% vs league 53.2% (+20.0%), 153 swings
  low: whiff 26.4% vs league 19.8% (+6.7%), 420 swings
  middle: whiff 14.7% vs league 12.3% (+2.4%), 464 swings
  high: whiff 29.6% vs league 18.1% (+11.4%), 159 swings


In [4]:
def pitching_approach(batter_id):
    """Recommended approach, generated ONLY from measured gaps.

    Every sentence must trace to a number that cleared its sample
    threshold. No sentence is produced from an unmeasured claim.
    """
    recs = []

    try:
        bp = by_pitch.loc[batter_id]
        bp = bp[bp["swings"] >= MIN_SWINGS_PER_PITCH].join(league)
        bp["gap"] = bp["whiff_pct"] - bp["lg_whiff"]
    except KeyError:
        return ["INSUFFICIENT SAMPLE"]

    if len(bp) == 0:
        return ["INSUFFICIENT SAMPLE"]

    # Best pitch to throw: largest whiff gap above league
    weak = bp[bp["gap"] > 0.05].sort_values("gap", ascending=False)
    for pt, r in weak.head(2).iterrows():
        recs.append(
            f"Attack with {pt}: whiffs {r['gap']:+.1%} above league "
            f"({r['whiff_pct']:.1%} on {int(r['swings'])} swings)")

    # Pitches to avoid
    strong = bp[bp["gap"] < -0.03].sort_values("gap")
    for pt, r in strong.head(2).iterrows():
        recs.append(
            f"Avoid {pt}: whiffs {r['gap']:+.1%} below league "
            f"({r['whiff_pct']:.1%} on {int(r['swings'])} swings)")

    # Location
    try:
        bz = by_zone.loc[batter_id]
        bz = bz[bz["swings"] >= MIN_SWINGS_PER_ZONE].copy()
        bz["gap"] = bz["whiff_pct"] - lg_zone.reindex(bz.index)
        best = bz["gap"].idxmax()
        if bz.loc[best, "gap"] > 0.05:
            recs.append(
                f"Work {best}: whiffs {bz.loc[best, 'gap']:+.1%} above league "
                f"({int(bz.loc[best, 'swings'])} swings)")
    except (KeyError, ValueError):
        pass

    if not recs:
        recs.append("No significant deviations from league average at these sample sizes")

    return recs

for pid in [592450, 665742, 660271]:
    print(f"=== {names.get(pid, pid)} ===")
    for r in pitching_approach(pid):
        print(f"  - {r}")
    print()

=== Judge, Aaron ===
  - Attack with CH: whiffs +16.5% above league (45.8% on 118 swings)
  - Attack with CU: whiffs +14.7% above league (44.3% on 61 swings)
  - Work below: whiffs +20.0% above league (153 swings)

=== Soto, Juan ===
  - Avoid CU: whiffs -15.8% below league (13.7% on 51 swings)
  - Avoid SL: whiffs -9.2% below league (23.1% on 134 swings)

=== Ohtani, Shohei ===
  - Attack with FF: whiffs +5.0% above league (24.0% on 384 swings)
  - Avoid FS: whiffs -4.1% below league (28.6% on 63 swings)
  - Work below: whiffs +18.5% above league (138 swings)



In [5]:
# How many batters get a real report vs INSUFFICIENT SAMPLE?
counts = by_pitch[by_pitch["swings"] >= MIN_SWINGS_PER_PITCH].groupby(level=0).size()
print(f"batters with at least one qualifying pitch type: {len(counts)}")
print(counts.describe().round(1).to_string())
print()
print("distribution of qualifying pitch types per batter:")
print(counts.value_counts().sort_index().to_string())

batters with at least one qualifying pitch type: 486
count    486.0
mean       4.5
std        2.2
min        1.0
25%        3.0
50%        5.0
75%        6.0
max        8.0

distribution of qualifying pitch types per batter:
1     84
2     33
3     46
4     55
5     61
6    103
7     72
8     32
